## Objetivo

Para la predicción financiera se han usado modelos que han estado presente por mucho tiempo en el mercado (ARIMA, SARIMA, GARCH, etc.). A estos modelos se les conoce como *Pronósticos de Series de Tiempo* (Time Series Forecasting) dado que tratan con historial de datos donde el tiempo les da su estructura. Estos modelos analizan las tendencias, la estacionalidad y el ruido para aprender los patrones inherentes en el tiempo y hacer la mejor estimación.

Sin embargo, podemos transformar los datos históricos de tal forma que funcione como entrenamiento para nuestros modelos de aprendizaje supervizado y esta transformación de la data se le conoce como *Sliding Window*.

En éste Juppyter Notebook presentaré una manera de usar *Sliding Window* para crear la **Tabla Analítica de Datos** que servirá para alimentar a nuestros modelos. Para esto definiré lo siguiente:

* Ventadas de Observación (vobs): El pasado que el modelo puede ver para aprender patrones. En este caso será de 15 días.
* Ventana de Desempeño (vdes): Es el futuro que quieres predecir. Esta será la variable objetivo. En este caso 1 día.
* Step (stride): Es cuánto se desplaza la ventana cada vez. Controla los pasos que genera y define cuánto se superponen las ventanas. En este caso 3 días de desplazamiento. 

## Dependencias


In [55]:
import yfinance as yf
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit


from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns
import cufflinks as cf
import plotly.express as px


from functools import reduce
from dateutil.relativedelta import relativedelta as rd

import warnings
import os 

warnings.filterwarnings('ignore')
cf.go_offline()
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', '{:,.4f}'.format)

## Lectura de Datos

In [56]:
df = yf.download(
    'KO',
    start='2020-01-01',
    end='2026-03-10'
)

df.columns = [c[0] for c in df.columns]

df.head()

[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,45.4327,45.7962,45.2427,45.7054,11867700
2020-01-03,45.1849,45.4327,44.6891,44.8792,11354500
2020-01-06,45.1683,45.3666,45.0444,45.1518,14698300
2020-01-07,44.8213,45.1105,44.7387,44.9866,9973900
2020-01-08,44.9039,45.1435,44.7387,44.8379,10676000


In [57]:
fig = px.line(
    df,
    x=df.index,
    y='Close',
    title='Precio de Cierre de Google'
)

fig.update_layout(
    xaxis_title = 'Fecha',
    yaxis_title = 'Precio de Cierre',
    template="plotly_white"
)

fig.show()

## Definición del Proyecto

In [58]:
# Redefinimos el index

df.reset_index(inplace=True)
display(df.head(3))

# Cambiamos el formato de Date para poder operar con fechas

df['Date'] = df['Date'].dt.date

,Date,Close,High,Low,Open,Volume
0,2020-01-02,45.4327,45.7962,45.2427,45.7054,11867700
1,2020-01-03,45.1849,45.4327,44.6891,44.8792,11354500
2,2020-01-06,45.1683,45.3666,45.0444,45.1518,14698300


In [59]:
# Para modelar, sólo usaré dos años de datos

fhi, fhf = df['Date'].max() + rd(years=-2), df['Date'].max()

print(f"La fecha inicial es {fhi} y la fecha final es {fhf}")

La fecha inicial es 2024-03-09 y la fecha final es 2026-03-09


In [60]:
# Limitamos la data

df = df.loc[df['Date']>=fhi].reset_index(drop = True).copy()

# Feature Engineering

Se hará ingeniería de variables para poder darle más información al modelo de donde aprender. Para esto, se aplicará la siguiente ingeniería:

* Tendencia (SMA/EMA):
    * *SMA*: Promedio siemple.
    * *EMA*: Promedio exponencial.

* Momentum
    * *Retornos*: Cambio porcentual del precio

* Momentum acumulado:
    * Retorno total en una ventana.

* Riesgo (Volatilidad Rolling):
    * ¿Qué tan bruscos son los movimientos?

* Rango verdadero promedio (ATR):
    * Rando diario High-Low promedio.

* RSI (Relative Strength Index):
    * Fuerza del movimiento reciente (0-100)

* MACD (Cambio de Tendencia):
    * Relación entre medias rápidas y lentas.

* Calendario (Estacionalidad):
    * Patrones repetitivos por fecha

* Soporte y Resistencia:
    * Niveles donde el precio suele repotar.

* Volumen Relativo:
    * Actividad vs promedio.

In [61]:
# Simple media average

df['sma_5'] = df['Close'].rolling(5).mean().shift(1)
df['sma_10'] = df['Close'].rolling(window=10).mean().shift(1)
df['sma_20'] = df['Close'].rolling(window=20).mean().shift(1)

# Exponential media average

df['ema_10'] = df['Close'].ewm(span=10).mean().shift(1)
df['ema_20'] = df['Close'].ewm(span=20).mean().shift(1)

# Derivados

df['dist_sma10'] = df['Close'] / df['sma_10'] - 1
df['sma_cross'] = df['sma_5'] - df['sma_20']

df.head(20)

,Date,Close,High,Low,Open,Volume,sma_5,sma_10,sma_20,ema_10,ema_20,dist_sma10,sma_cross
0,2024-03-11,56.3918,56.4386,55.7739,55.9612,14114300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-12,56.6352,56.8692,56.3637,56.4667,12684600,NaN,NaN,NaN,56.3918,56.3918,NaN,NaN
2,2024-03-13,57.2155,57.2998,56.9160,56.9909,13909500,NaN,NaN,NaN,56.5256,56.5195,NaN,NaN
3,2024-03-14,57.0882,57.3713,56.9938,57.1637,13996600,NaN,NaN,NaN,56.8030,56.7751,NaN,NaN
4,2024-03-15,56.5031,57.0410,56.2767,56.6352,36849900,NaN,NaN,NaN,56.8969,56.8655,NaN,NaN
5,2024-03-18,56.7390,56.9938,56.4276,56.5126,15856700,56.7668,NaN,NaN,56.7839,56.7778,NaN,NaN
6,2024-03-19,56.8334,56.9466,56.6730,56.8428,15030600,56.8362,NaN,NaN,56.7722,56.7697,NaN,NaN
7,2024-03-20,57.3241,57.3807,56.7673,56.7862,15258800,56.8759,NaN,NaN,56.7870,56.7817,NaN,NaN
8,2024-03-21,57.0599,57.5505,56.9183,57.1448,13067100,56.8976,NaN,NaN,56.9092,56.8755,NaN,NaN
9,2024-03-22,57.0787,57.3618,57.0221,57.1070,11502800,56.8919,NaN,NaN,56.9419,56.9050,NaN,NaN


In [62]:
# Momentum

df['retorno_1'] = df['Close'].pct_change(1)
df['retorno_5'] = df['Close'].pct_change(5)
df['retorno_10'] = df['Close'].pct_change(10)
df['retorno_20'] = df['Close'].pct_change(20)

# Momentum Acumulado
df['mom_10'] = df['Close'] / df['Close'].shift(10) -1
df['mom_20'] = df['Close'] / df['Close'].shift(20) -1

In [63]:
# Riesgo

df['risk_5'] = df['retorno_5'].rolling(5).std().shift(1)
df['risk_10'] = df['retorno_10'].rolling(10).std().shift(1)
df['risk_20'] = df['retorno_20'].rolling(20).std().shift(1)

# Rango entre el precio más alto y el más bajo

df['range'] = df['High'] - df['Low']
df['range_roll_14'] = df['range'].rolling(14).mean().shift(1)

In [64]:
# RSI correcto
delta = df['Close'].diff()
ganancia = delta.clip(lower=0)
perdida = -delta.clip(upper=0)

rs = ganancia.rolling(14).mean() / (perdida.rolling(14).mean() + 1e-9)
df['rsi_14'] = 100 - (100 / (1 + rs))

In [65]:
# MACD

ema12 = df['Close'].ewm(span=12).mean()
ema26 = df['Close'].ewm(span=26).mean()

df['macd'] = ema12 - ema26
df['macd_signal'] = df['macd'].ewm(span=9).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

In [66]:
# Calendario
df['day'] = pd.to_datetime(df['Date']).dt.dayofweek
df['month'] = pd.to_datetime(df['Date']).dt.month

In [67]:
# Soporte 

df['roll_max_20'] = df['High'].rolling(20).max().shift(1)
df['roll_min_20'] = df['Low'].rolling(20).min().shift(1)

df['dist_resistance'] = df['Close'] / df['roll_max_20'] - 1
df['dist_support'] = df['Close'] / df['roll_min_20'] - 1

In [68]:
# Volumen 
df['vol_sma_10'] = df['Volume'].rolling(10).mean()
df['vol_ratio'] = df['Volume'] / df['vol_sma_10']

In [69]:
df.to_csv("df.csv", index=False)